# Trigram Language Model (Neural Network Approach)

This notebook implements a character-level trigram language model using a single-layer neural network in PyTorch. The model predicts the next character based on the previous two characters.

In [ ]:
import torch
import torch.nn.functional as F

# Load dataset
names = open("names.txt", "r").read().splitlines()

# Character mappings
chars = sorted(list(set("".join(names))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)
print(f"Vocabulary size: {vocab_size}")

In [ ]:
# Create training set (xs: context of 2 chars, ys: target 3rd char)
xs, ys = [], []
for name in names:
    name = ".." + name + "."
    for ch1, ch2, ch3 in zip(name, name[1:], name[2:]):
        xs.append([stoi[ch1], stoi[ch2]])
        ys.append(stoi[ch3])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num_examples = xs.shape[0]
print(f"Number of training examples: {num_examples}")

In [ ]:
# Prepare input features (One-hot and flatten)
xenc = F.one_hot(xs, num_classes=27).float()
xenc_flat = xenc.view(-1, 54)

# Initialize weights
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad=True)

# Optimization Loop
learning_rate = 50
for k in range(1000):
    # Forward pass
    logits = xenc_flat @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    
    # Loss calculation (NLL + Smoothing)
    loss = -probs[torch.arange(num_examples), ys].log().mean() + 0.01 * (W**2).mean()
    
    # Backward pass
    W.grad = None
    loss.backward()
    
    # Update
    W.data += -learning_rate * W.grad
    
    if (k + 1) % 100 == 0:
        print(f"Step {k+1:4d} | Loss: {loss.item():.4f}")

In [ ]:
# Sampling from the model
g = torch.Generator().manual_seed(2147483647)

for i in range(10):
    out = []
    ix1, ix2 = 0, 0
    while True:
        # Prepare single input vector
        x1_enc = F.one_hot(torch.tensor([ix1]), num_classes=27).float()
        x2_enc = F.one_hot(torch.tensor([ix2]), num_classes=27).float()
        x_input = torch.cat((x1_enc, x2_enc), dim=1)
        
        # Predict probabilities
        logits = x_input @ W
        p = F.softmax(logits, dim=1)
        
        # Sample next character
        ix3 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix3])
        
        if ix3 == 0: break
        
        # Shift window
        ix1, ix2 = ix2, ix3
        
    print("".join(out))